# LocaleLive — Raw Log Explorer

This notebook pulls every raw log line from the `localelive-backend` Lambda and lets you browse, search, and understand the full shape of the log stream before doing any structured analysis.

**Use this notebook to:**
- See exactly what a log line looks like for each agent
- Spot unexpected messages, stack traces, or Lambda platform lines
- Find cold-start events and REPORT/INIT durations
- Free-text search across all messages
- Understand log volume and density per log stream

**Log group:** `/aws/lambda/localelive-backend`  
**Region:** `us-east-1`

---
### Setup
```bash
pip install boto3 pandas matplotlib seaborn python-dotenv
```
Place `credentials.env` in the same directory as this notebook.

In [ ]:
import os
import re
import time
from datetime import datetime, timezone
from pathlib import Path

import boto3
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from dotenv import load_dotenv

load_dotenv(Path("credentials.env"))

AWS_ACCESS_KEY_ID     = os.environ["AWS_ACCESS_KEY_ID"]
AWS_SECRET_ACCESS_KEY = os.environ["AWS_SECRET_ACCESS_KEY"]
AWS_DEFAULT_REGION    = os.environ.get("AWS_DEFAULT_REGION", "us-east-1")
LOG_GROUP_NAME        = os.environ.get("LOG_GROUP_NAME", "/aws/lambda/localelive-backend")

logs = boto3.client(
    "logs",
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
    region_name=AWS_DEFAULT_REGION,
)

print(f"Connected — log group: {LOG_GROUP_NAME}")

## 1 — List available log streams

Each Lambda invocation container writes to its own stream. Multiple streams = multiple concurrent containers.

In [ ]:
# ── Time window ──────────────────────────────────────────────────────────────
LOOKBACK_HOURS = 24  # widen for older data

now_ms   = int(time.time() * 1000)
start_ms = now_ms - LOOKBACK_HOURS * 3600 * 1000
start_dt = datetime.fromtimestamp(start_ms / 1000, tz=timezone.utc)
print(f"Window: last {LOOKBACK_HOURS}h (from {start_dt.strftime('%Y-%m-%d %H:%M UTC')})")    

streams = []
kwargs = dict(logGroupName=LOG_GROUP_NAME, orderBy="LastEventTime", descending=True, limit=50)
while True:
    resp = logs.describe_log_streams(**kwargs)
    streams.extend(resp.get("logStreams", []))
    token = resp.get("nextToken")
    if not token or len(streams) >= 50:
        break
    kwargs["nextToken"] = token

# Keep only streams with events in the window
streams = [
    s for s in streams
    if s.get("lastEventTimestamp", 0) >= start_ms
]

stream_df = pd.DataFrame([
    {
        "stream": s["logStreamName"],
        "first_event": datetime.fromtimestamp(s["firstEventTimestamp"] / 1000, tz=timezone.utc),
        "last_event":  datetime.fromtimestamp(s["lastEventTimestamp"]  / 1000, tz=timezone.utc),
        "stored_bytes": s.get("storedBytes", 0),
    }
    for s in streams
])
print(f"{len(stream_df)} active stream(s) in window")
stream_df

## 2 — Pull all raw log events

Fetches every line across all streams and parses the Lambda-prefixed format:
```
TIMESTAMP  LEVEL  logger_name  message
```
Platform lines (`START`, `END`, `REPORT`, `INIT_START`) are kept as a separate category.

In [ ]:
MAX_EVENTS_PER_STREAM = 2000  # raise if you want more history

all_events = []
for stream_name in stream_df["stream"].tolist():
    kwargs = dict(
        logGroupName=LOG_GROUP_NAME,
        logStreamName=stream_name,
        startTime=start_ms,
        endTime=now_ms,
        startFromHead=True,
        limit=200,
    )
    count = 0
    while count < MAX_EVENTS_PER_STREAM:
        resp = logs.get_log_events(**kwargs)
        batch = resp.get("events", [])
        if not batch:
            break
        for ev in batch:
            all_events.append({
                "timestamp_ms": ev["timestamp"],
                "timestamp":    datetime.fromtimestamp(ev["timestamp"] / 1000, tz=timezone.utc),
                "stream":       stream_name,
                "raw":          ev["message"].rstrip("\n"),
            })
        count += len(batch)
        token = resp.get("nextForwardToken")
        if not token or kwargs.get("nextToken") == token:
            break
        kwargs["nextToken"] = token

raw_df = pd.DataFrame(all_events).sort_values("timestamp").reset_index(drop=True)
print(f"Total events fetched: {len(raw_df)}")
raw_df[["timestamp", "stream", "raw"]].head(10)

## 3 — Parse and classify every line

In [ ]:
# Our app format:  2026-07-12 18:30:01 INFO     agents.google_maps  google_maps start ...
_APP_RE = re.compile(
    r"(?P<log_ts>\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2})\s+"
    r"(?P<level>DEBUG|INFO|WARNING|ERROR|CRITICAL)\s+"
    r"(?P<logger>\S+)\s+"
    r"(?P<message>.*)"
)

# Lambda platform lines
_PLATFORM_RE = re.compile(
    r"^(?P<kind>START|END|REPORT|INIT_START)\s+(?P<rest>.*)"
)

# rid= extractor
_RID_RE = re.compile(r"rid=(?P<rid>[\w-]+)")

def classify(raw: str) -> dict:
    m = _PLATFORM_RE.match(raw)
    if m:
        return {"category": "platform", "kind": m.group("kind"),
                "level": None, "logger": None, "message": raw, "rid": None}
    m = _APP_RE.search(raw)  # search not match — Lambda prepends a tab+request-id prefix
    if m:
        msg = m.group("message")
        rid_m = _RID_RE.search(msg)
        return {
            "category": "app",
            "kind":     m.group("level"),
            "level":    m.group("level"),
            "logger":   m.group("logger"),
            "message":  msg,
            "rid":      rid_m.group("rid") if rid_m else None,
        }
    return {"category": "other", "kind": "other",
            "level": None, "logger": None, "message": raw, "rid": None}

parsed = raw_df["raw"].apply(classify).apply(pd.Series)
df = pd.concat([raw_df, parsed], axis=1)

print(df["category"].value_counts().to_string())
print()
print(df[df["category"] == "app"]["level"].value_counts().to_string())

## 4 — Browse all unique message templates

Strips the variable parts (`rid=`, UUIDs, numbers) to show the distinct log event types the backend actually emits.

In [ ]:
app = df[df["category"] == "app"].copy()

def _template(msg: str) -> str:
    s = re.sub(r"rid=[\w-]+",          "rid=<id>",    msg)
    s = re.sub(r"\b[0-9a-f-]{36}\b",   "<uuid>",      s)
    s = re.sub(r"duration=[\d.]+s",    "duration=<s>",s)
    s = re.sub(r"results=\d+",         "results=<n>", s)
    s = re.sub(r"lat=[-\d.]+",         "lat=<f>",     s)
    s = re.sub(r"lon=[-\d.]+",         "lon=<f>",     s)
    s = re.sub(r"text='[^']*'",        "text='<q>'",  s)
    s = re.sub(r"query='[^']*'",       "query='<q>'", s)
    s = re.sub(r"\d+\.\d+",            "<f>",         s)
    s = re.sub(r"\b\d+\b",             "<n>",         s)
    return s.strip()

app["template"] = app["message"].apply(_template)

templates = (
    app.groupby(["level", "logger", "template"])
       .size()
       .reset_index(name="count")
       .sort_values(["level", "count"], ascending=[True, False])
)
pd.set_option("display.max_colwidth", 120)
templates

## 5 — Log volume over time

In [ ]:
if not df.empty:
    ts_df = df.set_index("timestamp").sort_index()
    hourly = ts_df.groupby([pd.Grouper(freq="30min"), "category"]).size().unstack(fill_value=0)

    fig, ax = plt.subplots(figsize=(14, 4))
    hourly.plot(kind="area", ax=ax, alpha=0.7,
                color={"app": "steelblue", "platform": "seagreen", "other": "coral"})
    ax.set_title("Log events per 30-minute bucket")
    ax.set_xlabel("UTC")
    ax.set_ylabel("Event count")
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d %Hh"))
    plt.xticks(rotation=35)
    plt.tight_layout()
    plt.show()

## 6 — Log level breakdown by logger

In [ ]:
if not app.empty:
    pivot = (
        app.groupby(["logger", "level"])
           .size()
           .unstack(fill_value=0)
           .reindex(columns=["DEBUG", "INFO", "WARNING", "ERROR", "CRITICAL"], fill_value=0)
    )
    # Sort by total event count descending
    pivot = pivot.loc[pivot.sum(axis=1).sort_values(ascending=False).index]

    fig, ax = plt.subplots(figsize=(13, max(3, len(pivot) * 0.5)))
    pivot.plot(kind="barh", stacked=True, ax=ax,
               color={"DEBUG": "#aec7e8", "INFO": "#1f77b4",
                      "WARNING": "#ff7f0e", "ERROR": "#d62728", "CRITICAL": "#7f0000"})
    ax.set_title("Log events by logger and level")
    ax.set_xlabel("Count")
    ax.set_ylabel("")
    plt.tight_layout()
    plt.show()
    print(pivot)

## 7 — Lambda platform events: cold starts & billed duration

`REPORT` lines contain init duration (cold start) and billed duration from Lambda itself.

In [ ]:
platform = df[df["category"] == "platform"].copy()
print(platform["kind"].value_counts().to_string())

# Parse REPORT lines
_REPORT_RE = re.compile(
    r"Duration:\s*(?P<dur>[\d.]+)\s*ms.*?"
    r"Billed Duration:\s*(?P<billed>[\d.]+)\s*ms.*?"
    r"Memory Size:\s*(?P<mem_size>\d+)\s*MB.*?"
    r"Max Memory Used:\s*(?P<mem_used>\d+)\s*MB"
    r"(?:.*?Init Duration:\s*(?P<init>[\d.]+)\s*ms)?",
    re.S
)

reports = []
for _, row in platform[platform["kind"] == "REPORT"].iterrows():
    m = _REPORT_RE.search(row["raw"])
    if m:
        reports.append({
            "timestamp":      row["timestamp"],
            "duration_ms":    float(m.group("dur")),
            "billed_ms":      float(m.group("billed")),
            "mem_size_mb":    int(m.group("mem_size")),
            "mem_used_mb":    int(m.group("mem_used")),
            "init_ms":        float(m.group("init")) if m.group("init") else None,
            "cold_start":     m.group("init") is not None,
        })

rep_df = pd.DataFrame(reports)
if not rep_df.empty:
    cold = rep_df[rep_df["cold_start"]]
    warm = rep_df[~rep_df["cold_start"]]
    print(f"\nTotal invocations: {len(rep_df)}  |  Cold starts: {len(cold)}  |  Warm: {len(warm)}")
    print(f"Cold start init p50: {cold['init_ms'].median():.0f}ms" if not cold.empty else "No cold starts")
    print(f"Billed duration  — p50: {rep_df['billed_ms'].quantile(0.5):.0f}ms  "
          f"p95: {rep_df['billed_ms'].quantile(0.95):.0f}ms")
    print(f"Memory used      — p50: {rep_df['mem_used_mb'].quantile(0.5):.0f}MB  "
          f"max: {rep_df['mem_used_mb'].max()}MB / {rep_df['mem_size_mb'].iloc[0]}MB")
    rep_df.tail(5)

In [ ]:
if not rep_df.empty and len(rep_df) >= 3:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    # Billed duration histogram, cold vs warm
    if not warm.empty:
        axes[0].hist(warm["billed_ms"], bins=30, label="warm", alpha=0.75, color="steelblue")
    if not cold.empty:
        axes[0].hist(cold["billed_ms"], bins=10, label="cold start", alpha=0.85, color="crimson")
    axes[0].set_title("Billed duration (ms): cold vs warm")
    axes[0].set_xlabel("ms")
    axes[0].set_ylabel("Count")
    axes[0].legend()

    # Memory used over time
    axes[1].scatter(rep_df["timestamp"], rep_df["mem_used_mb"],
                    c=["crimson" if c else "steelblue" for c in rep_df["cold_start"]],
                    s=25, alpha=0.8)
    axes[1].axhline(rep_df["mem_size_mb"].iloc[0], color="orange",
                    linestyle="--", linewidth=1, label=f"limit {rep_df['mem_size_mb'].iloc[0]}MB")
    axes[1].set_title("Memory used per invocation")
    axes[1].set_xlabel("UTC")
    axes[1].set_ylabel("MB used")
    axes[1].legend()
    plt.tight_layout()
    plt.show()

## 8 — WARNING and ERROR messages in full

In [ ]:
problems = app[app["level"].isin(["WARNING", "ERROR", "CRITICAL"])].copy()
print(f"{len(problems)} WARNING/ERROR/CRITICAL events")

if not problems.empty:
    pd.set_option("display.max_colwidth", 200)
    display(
        problems[["timestamp", "level", "logger", "message", "rid"]]
        .sort_values("timestamp")
        .reset_index(drop=True)
    )

## 9 — Free-text search across all messages

Set `SEARCH` to any string or regex to filter the raw log.

In [ ]:
SEARCH = "no_results"  # change to any keyword, e.g. "google_maps", "ERROR", "rid=abc"

mask = df["raw"].str.contains(SEARCH, case=False, regex=True, na=False)
hits = df[mask][["timestamp", "stream", "raw"]]
print(f"{len(hits)} lines matching {SEARCH!r}")
pd.set_option("display.max_colwidth", 300)
hits.reset_index(drop=True)

## 10 — Inspect one full log stream

Print every line from a single container invocation in time order.

In [ ]:
# Auto-picks the most recent stream; change to any name from section 1
STREAM = stream_df["stream"].iloc[0] if not stream_df.empty else ""

if STREAM:
    stream_events = df[df["stream"] == STREAM].sort_values("timestamp")
    print(f"Stream: {STREAM}  ({len(stream_events)} events)")
    print()
    for _, row in stream_events.iterrows():
        ts = row["timestamp"].strftime("%H:%M:%S.%f")[:-3]
        lvl = f"[{row['level']}]" if row["level"] else ""
        print(f"{ts}  {lvl:<10}  {row['raw'][:200]}")

## 11 — Export full parsed log to CSV

In [ ]:
out = Path("localelive_raw_log.csv")
export_cols = ["timestamp", "stream", "category", "level", "logger", "message", "rid"]
df[export_cols].to_csv(out, index=False)
print(f"Exported {len(df)} rows to {out}")